<a href="https://colab.research.google.com/github/Evelyn-Rojas/Simulacion-ll/blob/main/Proyecto2doParcial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#<font color="navy">**OPCIONES SOBRE FUTUROS**

---



En México, la producción de polímeros y plásticos depende de la refinación de petróleo, utilizando principalmente la nafta y el propileno derivados de los aproximadamente 1.8 millones de barriles diarios de crudo producidos en el país. Como se sabe, el precio del petróleo es muy volátil debido a diversos motivos, por lo que el uso de opciones de compra para una empresa que requiera petróleo como  materia prima es fundamental.

Supongamos que una empresa requiere de 100 mil barriles para su producción mensual. Usando precios históricos, proponer el precio de ejercicio y estimar cuánto se debería pagar por una opción de tipo call por cada barril de petróleo considerando un tiempo de maduración de un mes.

---

<font color="royalblue"> **Propuesta de Cobertura: Opción Call Europea**

Para una empresa que consume petróleo, la Opción Call es la herramienta ideal: otorga el derecho (pero no la obligación) de comprar a un precio fijo, protegiendo contra subidas de precio, pero permitiendo aprovechar bajadas si el mercado se estabiliza.

<font color="royalblue"> **Estimación de la Prima (Black-Scholes)**

Utilizando el modelo de Black-Scholes para opciones europeas, calculamos el costo por barril.

$$d_1 = \frac{\ln(S_0/K) + (r + \sigma^2/2)T}{\sigma\sqrt{T}}$$

$$d_2 = d_1 - \sigma\sqrt{T}$$

$$Call = S_0N(d_1) - Ke^{-rT}N(d_2)$$

El modelo Black-Scholes es el estándar para estimar el precio teórico de opciones financieras. Su relevancia radica en que transforma variables de mercado —precio spot, volatilidad, tasa libre de riesgo y tiempo— en una metodología matemática sólida para la gestión de riesgos.

Sus principales ventajas incluyen:

* Eficiencia operativa: Permite una implementación computacional rápida para obtener valuaciones consistentes.

* Análisis de sensibilidad: Facilita el cálculo de las "griegas" (como el Delta), permitiendo predecir cómo cambiará el valor del contrato ante fluctuaciones del activo subyacente.

* Cobertura estratégica: Es una herramienta crítica para proteger a las empresas contra la volatilidad en materias primas estratégicas, como el petróleo.

<font color="royalblue">**Programa aplicativo:**

In [20]:
import numpy as np
import pandas as pd
import yfinance as yf
from scipy.stats import norm

In [21]:
# WTI Crude Oil Futures
ticker = "CL=F"
# Datos históricos
datos = yf.download(ticker, period="2y")
# Precios de cierre
precios = datos["Close"].squeeze().dropna()

/tmp/ipykernel_651/4291867945.py:4: FutureWarning: YF.download() has changed argument auto_adjust default to True
  datos = yf.download(ticker, period="2y")
[*********************100%***********************]  1 of 1 completed


In [22]:
# Retornos logarítmicos
retornos = np.log(precios / precios.shift(1)).dropna()
# Volatilidad anualizada
volatilidad = float(retornos.std() * np.sqrt(252))

In [23]:
# Media y desviación histórica
media = float(precios.mean())
desv = float(precios.std())
# Strike propuesto
K_ejercicio = float(media + 0.5 * desv)

In [24]:
def black_scholes_call(S, K, T, r, sigma):
    #S: Precio actual (Spot)
    #K: Precio de ejercicio (Strike)
    #T: Tiempo a maduración (en años)
    #r: Tasa libre de riesgo (decimal)
    #sigma: Volatilidad implícita (decimal)
    d1 = (np.log(S / K)+ (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    call_price = (
        S * norm.cdf(d1)
        - K * np.exp(-r * T) * norm.cdf(d2)
    )
    return float(call_price), float(d1), float(d2)

**Parámetros del caso propuesto:**

In [25]:
# Precio spot actual
S_actual = float(precios.iloc[-1])
# Tiempo a vencimiento (1 mes)
T_mes = 1 / 12
# Tasa libre de riesgo (CETES aprox.)
r_mex = 0.0645

In [26]:
precio_opcion, d1, d2 = black_scholes_call(S_actual, K_ejercicio, T_mes, r_mex, volatilidad)

In [27]:
precio_opcion = float(precio_opcion)
d1 = float(d1)
d2 = float(d2)
print("=" * 55)
print("ANÁLISIS DE OPCIÓN CALL PETRÓLEO (WTI)")
print("=" * 55)

print(f"\nPrecio actual del WTI: ${S_actual:.2f} USD")
print(f"Precio de ejercicio (Strike): "
      f"${K_ejercicio:.2f} USD")

print(f"Volatilidad histórica anualizada: "
      f"{volatilidad:.4f}")
print(f"Precio del barril (Prima): "
      f"${precio_opcion:.2f} USD")
print(f"Costo total por 100,000 barriles: "
      f"${precio_opcion * 100000:,.2f} USD")

print(f"Delta (Probabilidad aproximada de ejercicio): "
      f"{norm.cdf(d1):.4f}")

ANÁLISIS DE OPCIÓN CALL PETRÓLEO (WTI)

Precio actual del WTI: $97.00 USD
Precio de ejercicio (Strike): $76.73 USD
Volatilidad histórica anualizada: 0.4279
Precio del barril (Prima): $20.78 USD
Costo total por 100,000 barriles: $2,078,294.89 USD
Delta (Probabilidad aproximada de ejercicio): 0.9774


In [28]:
# ============================================
# ESCENARIOS DE AHORRO
# ============================================

precios_escenario = [90, 100, 110, 120, 130]

print("\n" + "=" * 65)
print("ANÁLISIS DE ESCENARIOS DE COBERTURA")
print("=" * 65)

for precio_final in precios_escenario:

    # Sin cobertura
    costo_sin = precio_final * 100000

    # Con cobertura
    # Si conviene ejercer:
    if precio_final > K_ejercicio:

        costo_con = (K_ejercicio * 100000+ precio_opcion * 100000)

    # Si NO conviene ejercer:
    else:
        costo_con = (precio_final * 100000+ precio_opcion * 100000)
    ahorro = costo_sin - costo_con
    print(f"\nPrecio final WTI: ${precio_final:.2f}")
    print(f"Costo sin cobertura: "
          f"${costo_sin:,.2f}")
    print(f"Costo con cobertura: "
          f"${costo_con:,.2f}")
    print(f"Ahorro estimado: "
          f"${ahorro:,.2f}")


ANÁLISIS DE ESCENARIOS DE COBERTURA

Precio final WTI: $90.00
Costo sin cobertura: $9,000,000.00
Costo con cobertura: $9,751,674.80
Ahorro estimado: $-751,674.80

Precio final WTI: $100.00
Costo sin cobertura: $10,000,000.00
Costo con cobertura: $9,751,674.80
Ahorro estimado: $248,325.20

Precio final WTI: $110.00
Costo sin cobertura: $11,000,000.00
Costo con cobertura: $9,751,674.80
Ahorro estimado: $1,248,325.20

Precio final WTI: $120.00
Costo sin cobertura: $12,000,000.00
Costo con cobertura: $9,751,674.80
Ahorro estimado: $2,248,325.20

Precio final WTI: $130.00
Costo sin cobertura: $13,000,000.00
Costo con cobertura: $9,751,674.80
Ahorro estimado: $3,248,325.20


<font color="royalblue"> **Conclusión:**

La valuación de la opción call sobre petróleo WTI mediante el modelo de Modelo Black-Scholes permitió estimar el costo de cobertura para una empresa que utiliza petróleo como materia prima. Utilizando datos históricos se calculó la volatilidad del activo y se propuso un precio de ejercicio basado en estadísticas históricas, obteniendo una estimación más objetiva del contrato de cobertura.

Los resultados muestran que la empresa debe pagar una prima para asegurar el derecho de comprar petróleo a un precio fijo durante el siguiente mes, reduciendo así el riesgo ante posibles aumentos en el precio del crudo. Además, el modelo permitió analizar la sensibilidad de la opción mediante el Delta y relacionar conceptos estadísticos y financieros aplicados a la gestión de riesgos.

Aunque el modelo Black-Scholes simplifica ciertas condiciones del mercado real, representa una herramienta útil para la toma de decisiones financieras y la protección ante la volatilidad del mercado petrolero.